# 06 - Split (portfolios)

Takes the odds capture form written by `05_Run`, once the prices are in, and
turns it into a set of portfolios to choose between.

The pipeline, in order:

1. **Best price** -- `O = max` over the books in `staking.BOOKS`. A proposition
   priced by no book is dropped. `Book` names the winner, or every winner on a
   tie.
2. **Edge** -- `E = P x O`. Above 1 the book pays more than the model says it
   should.
3. **Discard `E < 1`.** An event can lose everything here; that is normal.
4. **Remove dominated propositions, within each event** -- anything another beats
   on *both* `P` and `O`. Not `P` and `E`: `E` is `P x O`, so comparing against it
   counts the probability twice. Exact ties on both survive.
5. **Search the portfolios** -- every selection of one proposition from each of
   any subset of events, scored under both the `1/E` split and the
   minimum-variance split.
6. **Keep the undominated ones** -- nothing else beats them on expected return,
   variance *and* probability of profit at once.

There is no step 7 and no final list. At portfolio level the three criteria trade
against each other and there is no single best answer, so what comes out is a set
to filter, in the notebook or on the workbook's Query sheet.

This notebook contains no logic of its own; everything lives in `fpp.portfolio`.

In [1]:
import fpp
import numpy as np
import pandas as pd

from fpp import portfolio as pf

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
print("fpp", fpp.__version__)

fpp 0.1.0


## 1. Inputs

`ODDS_FILE` **resolves to the newest form on disk** rather than being typed. It
used to be a hardcoded date, and that is exactly how a run two days later
consumed a two-day-old form: fixtures that were no longer on, priced by a model
that had since been retuned, and nothing anywhere to notice. The form carries its
own probabilities and `06_Split` reads nothing else, so there is no second source
that could have disagreed.

`check_form_is_current` raises if the file is not the newest one. Pin a specific
form deliberately if you mean to re-price an old one.

`STAKE` seeds cell B1 of the calculator -- it stays editable in Excel afterwards,
and every percentage is independent of it, so this number only sets the cash
column.

`LEG_VAR` is **how many events you are willing to leave out**, and it is the
skipping switch. `0` requires one bet from every qualifying event, which is the
only setting whose answer is provable -- the space collapses to something that
enumerates, so every portfolio is scored rather than searched. Each rung above
that opens the search one event wider.

It is deliberately *relative*, and that is the whole point of the control. How
many events qualify is not knowable when you set it: a 45-fixture weekend might
yield 43 events or 37, depending on which markets got priced and which cleared
`e >= 1`. Typing an absolute floor against a guess of 43 quietly becomes "skip
nothing" if only 38 qualify, and asks for the impossible if 36 do. `LEG_VAR`
says the thing actually meant and resolves against whatever the form turns out
to hold.

Keeping it small is worth doing, and not only for the reason below. Every
variance and probability figure here is computed *assuming the model
probabilities are right*; nothing accounts for them being wrong. A two-leg
portfolio can read 95% profitability while resting its whole edge on one
proposition being accurate to a point or two. Spreading over more events is the
only defence against that, and it is the one risk the numbers omit.

It is also the cheapest runtime lever there is. The search itself takes the same
time whatever the floor, but what comes out of it does not -- on a 19-event form,
431,244 states at no floor against 184,250 at `LEG_VAR = 5` and 30,593 at `0` --
and everything downstream is priced per portfolio.

In [2]:
ODDS_FILE = fpp.report.latest_form()      # newest odds_input_*.xlsx; pin a path to override
STAKE     = 220.0

LEG_VAR   = 2      # how many qualifying events a portfolio may leave out.
                   # 0 = one bet from every event, no skipping (and provable).
                   # Resolves against however many events actually qualify.

WRITE_WORKBOOK = False   # the Excel portfolio workbook. Superseded by the Edge
                         # Book app below; kept because it costs one line to keep
                         # and 5.7 MB a day to write.

## 2. Read the form

In [3]:
# Refuses a form that is not the newest on disk -- see the note above.
meta = fpp.report.check_form_is_current(ODDS_FILE)
print(f"form      : {meta['path'].name}")
print(f"generated : {meta['generated']}  from {meta['source']}")
print(f"fixtures  : {meta['fixtures']}")

filled = fpp.report.read_filled(ODDS_FILE)
display(filled.head(8))

form      : odds_input_2026-09-12.xlsx
generated : 2026-09-12  from predictions_2026-09-12.xlsx
fixtures  : 3
Read 222 propositions from odds_input_2026-09-12.xlsx (generated 2026-09-12 from predictions_2026-09-12.xlsx); 136 priced


,sheet_code,label,p,b365,paddypower,tenbet,boylesports,betmgm,virginbet,date,league,home_team,away_team
0,SP1-01,Goals - Athletic Club - Over 0.5,0.830873,1.083,1.083333,1.083,1.07,1.09,NaN,2026-09-12,La Liga,Athletic Club,Elche
1,SP1-01,Goals - Athletic Club - Over 1.5,0.530316,1.444,1.444444,1.480,1.44,1.48,NaN,2026-09-12,La Liga,Athletic Club,Elche
2,SP1-01,Goals - Athletic Club - Over 2.5,0.263256,2.500,2.625000,2.450,2.40,2.45,NaN,2026-09-12,La Liga,Athletic Club,Elche
3,SP1-01,Goals - Athletic Club - Over 3.5,0.105058,5.000,5.000000,NaN,5.00,4.70,NaN,2026-09-12,La Liga,Athletic Club,Elche
4,SP1-01,Goals - Elche - Over 0.5,0.578578,1.727,1.666667,1.700,1.67,1.68,NaN,2026-09-12,La Liga,Athletic Club,Elche
5,SP1-01,Goals - Elche - Over 1.5,0.214418,4.500,4.500000,4.500,4.33,4.20,NaN,2026-09-12,La Liga,Athletic Club,Elche
6,SP1-01,Goals - Elche - Over 2.5,0.057080,17.000,12.000000,NaN,13.00,10.00,NaN,2026-09-12,La Liga,Athletic Club,Elche
7,SP1-01,Goals - Elche - Over 3.5,0.011760,51.000,34.000000,NaN,34.00,13.00,NaN,2026-09-12,La Liga,Athletic Club,Elche


## 3. Qualify

Watch the funnel. `(P, O)` dominance prunes far less than the old `(P, E)` rule
did -- `P` and `O` are close to inverses of each other, so they correlate
strongly negatively and mutual dominance is rare. That is the rule working: the
survivors are genuinely incomparable, and choosing between them is what the
search is for.

In [4]:
qualified = pf.qualify(filled)
options   = pf.event_options(qualified)
min_legs  = options.min_legs_for(LEG_VAR)   # the same resolution `pf.search` will do

print(f"priced           : {int(filled[list(fpp.staking.BOOK_COLUMNS)].notna().any(axis=1).sum())}")
print(f"E >= 1, undominated: {len(qualified)} across {options.n_events} events")
print(f"options per event  : {options.sizes.tolist()}")
print(f"leg range          : {min_legs}-{options.n_events} legs  (LEG_VAR = {LEG_VAR})")
print(f"search space       : {options.space(min_legs, None):,.4g} portfolios")

priced           : 136
E >= 1, undominated: 34 across 3 events
options per event  : [17, 20, 21]
leg range          : 1-3 legs  (LEG_VAR = 2)
search space       : 8,315 portfolios


## 4. Search

Below `EXHAUSTIVE_MAX` every portfolio is enumerated; above it the dynamic
program walks the space instead, keeping the `(expected return, variance)`
frontier exactly and a band around it. `info["mode"]` says which happened, because
"all of them" and "the ones the search reached" are different claims.

In [5]:
res = pf.search(filled, leg_var=LEG_VAR)

info, scored = res["info"], res["scored"]
print(f"mode            : {info['mode']}  ({info['found']:,} reached -> pool {info['pool']:,})")
print(f"legs            : {info['min_legs']}-{info['max_legs']}  (leg_var = {info['leg_var']})")
print(f"rows scored     : {len(scored):,}")
print(f"undominated     : {int(scored['undominated'].sum()):,}")

mode            : exhaustive  (8,315 reached -> pool 8,315)
legs            : 1-3  (leg_var = 2)
rows scored     : 8,315
undominated     : 271


## 5. The undominated set

Every row here is beaten by nothing else on all three of expected return,
variance and probability of profit. The extremes are worth looking at first --
they are what the trade-off actually costs.

In [6]:
# Read off `scored` rather than listed: `staking.THRESHOLDS` is meant to be tuned,
# and naming the columns by hand meant trimming it broke this cell rather than
# just narrowing the table. `median_leg_p` sits next to `legs` because the two
# answer the same question -- how many bets, and how likely each one is.
THRESH = [c for c in scored.columns if c.startswith("p_over_")]
VIEW = ["id", "split", "legs", "median_leg_p", "pct_expected_return", "pct_sd", *THRESH]
FMT  = {"pct_expected_return": "{:.2%}", "pct_sd": "{:.2%}",
        "median_leg_p": "{:.1%}", **{c: "{:.2%}" for c in THRESH}}

undominated = scored[scored["undominated"]]
if undominated.empty:
    print("Nothing survived -- either nothing was priced, or no price beat the model.")
else:
    for title, sub in (
        ("highest probability of profit", undominated.nlargest(5, "p_over_100")),
        ("highest expected return",       undominated.nlargest(5, "pct_expected_return")),
        ("lowest spread",                 undominated.nsmallest(5, "pct_sd")),
        ("likeliest legs",                undominated.nlargest(5, "median_leg_p")),
    ):
        print(f"\n--- {title} ---")
        display(sub[VIEW].style.format(FMT).hide(axis="index"))


--- highest probability of profit ---


id,split,legs,median_leg_p,pct_expected_return,pct_sd,p_over_90,p_over_100,p_over_110
34,Growth,2,46.1%,113.98%,133.74%,89.91%,89.91%,2.61%
35,Growth,2,45.4%,103.50%,68.48%,89.76%,89.76%,1.14%
22,Growth,1,89.6%,100.84%,34.28%,89.64%,89.64%,89.64%
472,Growth,2,47.3%,120.62%,95.01%,85.27%,85.27%,11.21%
704,Growth,2,45.9%,107.67%,59.51%,84.80%,84.80%,84.80%



--- highest expected return ---


id,split,legs,median_leg_p,pct_expected_return,pct_sd,p_over_90,p_over_100,p_over_110
11,Growth,1,5.6%,285.99%,1173.36%,5.61%,5.61%,5.61%
275,Growth,2,5.0%,240.02%,871.53%,9.74%,9.74%,9.74%
3707,Growth,2,9.2%,236.29%,522.01%,17.71%,17.71%,17.71%
3971,Growth,3,5.6%,225.32%,477.60%,21.31%,21.31%,21.31%
3706,Growth,2,12.0%,220.76%,422.38%,22.59%,22.59%,22.59%



--- lowest spread ---


id,split,legs,median_leg_p,pct_expected_return,pct_sd,p_over_90,p_over_100,p_over_110
509,Growth,3,80.5%,104.51%,32.84%,76.23%,67.17%,37.46%
531,Growth,3,78.3%,104.68%,33.65%,75.40%,65.30%,36.42%
22,Growth,1,89.6%,100.84%,34.28%,89.64%,89.64%,89.64%
484,Growth,2,86.5%,103.35%,35.29%,83.41%,74.77%,74.77%
519,Growth,3,80.5%,104.64%,36.48%,67.55%,67.55%,67.55%



--- likeliest legs ---


id,split,legs,median_leg_p,pct_expected_return,pct_sd,p_over_90,p_over_100,p_over_110
22,Growth,1,89.6%,100.84%,34.28%,89.64%,89.64%,89.64%
484,Growth,2,86.5%,103.35%,35.29%,83.41%,74.77%,74.77%
462,Growth,1,83.4%,104.27%,46.50%,83.41%,83.41%,83.41%
513,Growth,3,80.5%,109.56%,48.11%,74.59%,74.59%,24.26%
507,Growth,3,80.5%,115.21%,44.85%,72.13%,69.80%,69.80%


## 6. Filter

`filter_portfolios` takes the same criteria as the workbook's Query sheet, as
`min_<column>` / `max_<column>`. Leave one out and that constraint is dropped.
There is no right answer here -- change the numbers until the shape of the
outcome is one you want.

In [7]:
picked = pf.filter_portfolios(
    scored,
    min_pct_expected_return=1.05,
    min_p_over_100=0.70,
    max_pct_sd=0.25,
    # The ledger's first settled slate ranked `median_leg_p` +0.72 against realised
    # return and `pct_expected_return` -0.76. One Saturday, so this is left off
    # rather than switched on -- but it is the constraint that evidence points at.
    # min_median_leg_p=0.55,
    sort_by="p_over_100",
)
print(f"{len(picked)} portfolios match")
display(picked[VIEW].head(15).style.format(FMT).hide(axis="index"))

0 portfolios match


id,split,legs,median_leg_p,pct_expected_return,pct_sd,p_over_90,p_over_100,p_over_110


## 7. One portfolio, as bets to place

Point this at any `id` from the tables above. `Book` travels with each leg
because an edge you cannot find again is not actionable.

In [8]:
if not scored.empty:
    row = (picked if len(picked) else undominated if len(undominated) else scored).iloc[0]
    bets = pf.legs(res["picks"], res["options"], int(row["combo"]), row["split"], total=STAKE)
    print(f"portfolio {int(row['id'])}  ({row['split']} split, {int(row['legs'])} legs)")
    print(f"  expected return {row['pct_expected_return']:.2%}   "
          f"sd {row['pct_sd']:.2%}   P(profit) {row['p_over_100']:.2%}")
    display(bets.style.format({"p": "{:.2%}", "o": "{:.2f}", "e": "{:.4f}",
                               "stake": "{:,.2f}"}).hide(axis="index"))

portfolio 34  (Growth split, 2 legs)
  expected return 113.98%   sd 133.74%   P(profit) 89.91%


event,fixture,label,p,o,book,e,stake
SP1-02,Osasuna vs Espanyol,Shots on Target - Osasuna - Over 2.5,89.64%,1.12,10bet,1.0084,202.24
SP1-04,Real Madrid vs Rayo Vallecano,Shots on Target - Rayo Vallecano - Over 8.5,2.61%,101.00,Paddy Power,2.6351,17.76


## 8. Write the Edge Book

Two JSON files and one page. `portfolios.json` carries the **undominated set
only** -- 4,940 of 200,000 scored rows on a real form -- with each portfolio's
legs as an array of proposition ids rather than duplicated rows, which is what
lets a 5.7 MB workbook become roughly 3 MB of data.

`write_edge_book` fuses that with the `predictions.json` written by `05_Run` and
the app source in `app/edge-book/` into one self-contained HTML file. Open it
directly -- there is no server and nothing to install. It refuses to build if the
two JSON files are from different runs, because a page pairing yesterday's
fixtures with today's portfolios would open looking perfectly fine.

Set `WRITE_WORKBOOK = True` in section 1 to also write the old Excel workbook.

Where it all lands: the page goes to the `Outputs` root and pushes yesterday's
into `Outputs/Archive/edge_book/`, and the JSON goes to `Outputs/Data/`. The root
holds one predictions workbook, one odds form and one page — which is what lets
`latest_form()` and `latest_json()` answer "the current one" by position rather
than by reading dates off a listing.

`ledger.capture` writes this run into `Outputs/Analysis/Pending/` on the way
past. That has to happen here rather than in `07_Analysis`: `_write_json`
deletes yesterday's `portfolios_*.json` the instant today's lands, so a slate
nobody captured before the next run of this notebook is simply gone. The inbox
accumulates -- run `06` five times before `07` and five slates are waiting.

In [9]:
if WRITE_WORKBOOK:
    out = fpp.report.write_portfolio_workbook(res, stake=STAKE, source=ODDS_FILE.name)
    print("->", out)

# Capture first, because it is what names the run. Portfolio ids restart at 1
# every run, so `770` stops identifying anything the moment there are two of
# them; the code goes into the payload below and the page prints `R007-770`.
slate = fpp.ledger.capture(res, filled, source=ODDS_FILE.name)
print("->", slate)

# `filled` goes in as well as `res`: it carries every price a book quoted, not
# just the ones that cleared E >= 1, and the Match Board draws a negative edge
# bar wherever the model looked and the price was not there.
port_json = fpp.report.write_portfolios_json(res, filled, source=ODDS_FILE.name,
                                             run_code=slate.run_code)
book = fpp.report.write_edge_book()
print("\n->", book)

-> R015 -> /Users/patrickknott/Developer/proposition-portfolio/Outputs/Analysis/Pending/slate_R015_2026-09-12 (136 props, 271 portfolios)
Wrote 271 undominated portfolios of 8,315 scored, 34 propositions (136 priced) -> /Users/patrickknott/Developer/proposition-portfolio/Outputs/Data/portfolios_2026-09-12.json
Wrote 1.2 MB -> /Users/patrickknott/Developer/proposition-portfolio/Outputs/edge_book_2026-09-12.html

-> /Users/patrickknott/Developer/proposition-portfolio/Outputs/edge_book_2026-09-12.html
